# L13a: Synthetic Markets with Jump Hidden Markov Models
In this lecture, we build synthetic markets: simulated price histories that share the features of real ones, such as fat tails, near-zero autocorrelation, and volatility clustering. 

__Why synthetic markets?__ A backtest tests a trading strategy on one historical path. How would the same method do on thousands of plausible paths, including sell-offs that never happened?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct Markov chains and hidden Markov models:__ Define a state space, a transition matrix, and its stationary distribution, and describe how hidden states emit the values we observe.
> * __Build a jump hidden Markov model of one asset:__ Bin the daily growth rate into states, estimate the transition matrix by counting, fit an emission distribution in each state, and explain why runs of large daily moves (jump episodes) are needed to reproduce volatility clustering.
> * __Compose a synthetic market of many stocks:__ Combine each stock's baseline with market, sector, and stock-specific residual components to build synthetic stocks that move together like real ones.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * A Markov chain is a set of states $\mathcal{S}$ and a transition matrix $\mathbf{P}$ whose rows sum to one. Its stationary distribution satisfies $\bar{\pi} = \bar{\pi}\mathbf{P}$.
> * The jump hidden Markov model bins the daily growth rate into states of equal probability, estimates $\mathbf{P}$ by counting, and draws each day's growth rate from a Student-t distribution of the day's state. Jump episodes (runs of large daily moves) start with probability $\epsilon$ and last $\lambda$ days on average, allowing this approach to reproduce volatility clustering.
> * We model a synthetic stock's growth rate as $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,\tilde{\varepsilon}_{i}$, with a stock-specific baseline $\alpha_{i}$. Shared market and sector components capture correlations between stocks. The residual draw $\tilde{\varepsilon}_{i}$ is centered on zero and represents the movement left after subtracting the baseline, market, and sector contributions.
>
>   The residual scale $\psi_{i}$ targets the stock's volatility. A drifting $\beta_{i}$ lets its market sensitivity change over time.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The fitting and sampling algorithms of the jump hidden Markov model, step by step.
> * How periods of high volatility develop and fade in simulated markets compared with market data.
> * How the sector loadings and the scale of the residual component are chosen, and the drifting $\beta$ option.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13a) lists this lecture's symbols with their meanings and units.

Let's get started!

___

## Examples
__We will work through these examples in lecture:__

> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024. We compare simulated distributions and volatility clustering with the data, then tune jump episodes and examine what they change.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.

___

## Review: Wolfram Markets
Previously, we built Wolfram markets, in which each trader reads its four neighbors' intentions and uses a rule table to choose its next intention. The three rules below give the next intention for every neighbor sum:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-Wolfram-Rules.svg" width="700" style="max-width:100%; height:auto;" alt="A trader and its four neighbors on the left feed a neighbor sum of 1 into three numeric rule tables on the right, with columns from 8 to 0. Rules 19280 and 19286 differ only at sum 1, highlighted in yellow: the output changes from 0 (Buy) to 2 (Sell). The added Rule 0 row is all zeros (Buy)."/>
    </center>
</div>

R19280 and R19286 differ at just one neighbor sum, while R0 always buys. In the two switched markets below, 25% of traders switch to R19286 after step 1000.

In the bottom market, investors are randomly replaced at each step after step 1500, when a Sell majority has formed. On average, one quarter of the new arrivals follow R0.

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-Wolfram-Markets.svg" width="1000" style="max-width:100%; height:auto;" alt="Market grids at steps 1000, 1250, 1500, 1600, 2000, and 4000. Rows show R19280 alone, the closed rule-switch market, and the switched market with continuous refresh after step 1500. The switched rows share their history through step 1500, with 65 percent Sell intentions. R0 entry then breaks up the Sell majority in the bottom row."/>
    </center>
</div>

In the closed market, Sell intentions spread across the grid. The price falls until the sellers run out of shares. In the open market, new buyers break up the Sell majority and the price recovers.

__Today:__ A market can change its character without warning. Our goal is a trade management system that makes money when it can and, at a minimum, preserves cash when it cannot.

To reach that goal, the system has to adapt as the market changes. To test such a system, we need many plausible paths of a changing market.

___

## Company Profile: State Street
[State Street Investment Management](https://www.ssga.com/us/en/about-us) is the asset management business of State Street Corporation. Established in [1978](https://www.ssga.com/us/en/about-us/who-we-are/our-history), it was known as State Street Global Advisors until 2025.

> __What does the firm do?__
>
> * __Investment management:__ The firm manages index funds and active investment strategies. Its [SPDR ETF family includes SPY](https://www.ssga.com/us/en/institutional/etfs/state-street-spdr-sp-500-etf-trust-spy), launched in 1993 to track the S&P 500.
> * __Regime modeling:__ Market regimes are periods with different patterns of risk and return. A hidden Markov model treats the regime as an unobserved state and models transitions between regimes. In a [January 2026 article](https://www.ssga.com/us/en/institutional/insights/how-ai-is-transforming-investment-management-state-street-strategic-approach), State Street describes using these models to guide trading in commodities, credit, and broader markets as regimes change.

__Explore further:__

* __Jobs and internships:__ State Street's [early careers page](https://careers.statestreet.com/global/en/state-street-early-careers) describes student internships and graduate development programs. Browse [current openings](https://careers.statestreet.com/global/en/search-results) for available roles.
* __Video:__ Watch the firm's [A Day in the Life at State Street series](https://careers.statestreet.com/global/en/state-street-early-careers) for examples of employees' work.
* __Research:__ The 2026 article cites [Modeling Credit Spreads Through Regime Switching with Gradual Transition](https://doi.org/10.3905/jfi.2023.1.164), published in *The Journal of Fixed Income* in 2023.

__Connection to today's lecture:__ State Street uses hidden Markov models to inform investment decisions as market conditions change. Here, we use a hidden Markov model to generate synthetic price histories with changing behavior that we can use to test trading strategies.

___


## Markov Chains and Hidden Markov Models
A Markov chain is a stochastic (random) process that moves between states. The next state depends only on the present state. For example, the states could be the moods of an investor: bullish, neutral, or bearish.

> __Definition (Markov chain):__ A Markov chain has a set of states $\mathcal{S}=\left\{s_{1},\dotsc,s_{N}\right\}$ and a transition matrix $\mathbf{P}\in\mathbb{R}^{N\times{N}}$. Let $X_{t}$ be the state on day $t$. The element $p_{ij}\in\mathbf{P}$ is the probability of moving from $s_{i}$ today to $s_{j}$ tomorrow:
>
> $$
> p_{ij} = P\left(X_{t+1}=s_{j}\mid X_{t}=s_{i}\right)
> $$
>
> Each row of $\mathbf{P}$ sums to one and is a $\texttt{categorical}$ distribution over tomorrow's state. The __stationary distribution__ $\bar{\pi}$ solves $\bar{\pi} = \bar{\pi}\mathbf{P}$ and gives the long-run share of time the chain spends in each state.

We estimate the transition matrix $\mathbf{P}$ from data:

> __Proposition (estimating the transition matrix):__ In a sequence of observed states $X_{1},\dotsc,X_{T}$, let $n_{ij}$ count the days in state $s_{i}$ followed by a day in state $s_{j}$. The maximum likelihood estimate of $p_{ij}$ is the share of the days in state $s_{i}$ that moved to state $s_{j}$:
>
> $$
> \boxed{\hat{p}_{ij} = \frac{n_{ij}}{\sum_{k=1}^{N} n_{ik}}}
> $$
>
> This estimate maximizes the log-likelihood $\sum_{i,j} n_{ij}\ln p_{ij}$, subject to each row of $\mathbf{P}$ summing to one.

A __hidden Markov model__ adds observable outputs to the chain (unobservable) states. Each day, we observe an output (emission) drawn from a distribution that depends on that day's hidden state.

The [Markov models notebook](advanced/markov/CHEME-5660-L13a-Lecture-MarkovModels-Fall-2026.ipynb) covers the Markov property, power iteration for $\bar{\pi}$, and sampling a hidden Markov model.

___

## The Jump Hidden Markov Model
Geometric Brownian motion assumes normally distributed growth rates and constant volatility.

Real growth rates have __near-zero autocorrelation__: past growth rates offer little help in a linear forecast of the next growth rate. They have __fat tails__: large daily moves are far more common than a normal distribution predicts. They show __volatility clustering__: large moves of either sign tend to follow large moves.

Let's build a hidden Markov model of the daily growth rate $g_{t}$ (units: 1/yr) of one asset. It has three parts:
* __States:__ We fit a Laplace distribution to the growth rates and use its cumulative distribution function $F_{L}$ to cut it into $N$ slices of equal probability. The slices run from the most negative growth rates (state 1) to the most positive (state $N$).
* __Transition matrix:__ Each day of the data falls in one slice, which gives a sequence of states. We estimate $\hat{\mathbf{P}}$ by counting the moves between the states of consecutive days.
* __Emissions:__ In state $s_{k}$, the growth rate is $g_{t} \mid X_{t} = s_{k} \sim \mu_{k} + \sigma_{k}\,T_{\nu}$. We set $\mu_{k}$ and $\sigma_{k}$ to the mean and standard deviation of that state's growth rates. The Student-t variable $T_{\nu}$ has $\nu$ degrees of freedom, giving a draw standard deviation of about $1.3\sigma_{k}$ when $\nu=5$.

During fitting, we assign each observed growth rate to its slice. During simulation, the model draws a hidden state and reports only its emitted growth rate.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Fit-Schematic.svg" width="1000" style="max-width:100%; height:auto;" alt="The fit of the model in four labeled steps. On the left, a plot of the daily growth rate of a ticker on sixteen trading days, with five horizontal bands shaded behind the points. Each point is a small circle with the number of its state inside: 3, 4, 3, 2, 3, 4, 5, 1, 5, 1, 2, 3, 4, 3, 3, 2. The three moves from state 3 to state 4 on consecutive days are orange segments of the line, and a key in the corner reads a move 3 to 4. In the middle, under the heading 1. Laplace, the fitted Laplace distribution is drawn sideways on the same growth-rate axis, with its peak in the middle band. The band boundaries cut the area under the distribution into N = 5 slices of equal probability, each labeled 20 percent. A bar to the right, headed 2. States, names the state and the mood of each band: state 1, very bearish, in red at the bottom, then state 2, bearish, state 3, neutral, state 4, bullish, and state 5, very bullish, in blue at the top. An orange arrow labeled n i j leads from the bar to a table of counts headed 3. Count. The table has the state on day t in the rows and the state on day t plus 1 in the columns, and the cell in row 3 and column 4 holds 3 and is highlighted in orange. An arrow leads from the table to a box headed 4. Transitions, with the estimate of p i j, n i j divided by the sum of row i, and the example for row 3, 3 over 6."/>
    </center>
</div>

### Fitting algorithm

__Initialize__: Given the daily growth rates $g_{1},\dotsc,g_{T}$ of one ticker, the number of states $N$, and the degrees of freedom $\nu$ of the emissions. Set the counts $n_{ij}\gets 0$ for every pair of states $s_{i}$ and $s_{j}$.

1. __Laplace:__ Fit the Laplace distribution by maximum likelihood. Set the location $\mu$ to the median of the growth rates, and set the scale $b \gets \frac{1}{T}\sum_{t=1}^{T}\lvert g_{t} - \mu\rvert$.
2. __States:__ For each day $t = 1,2,\dots,T$ __do__:
    - a. Compute the cumulative probability $F_{L}(g_{t})$.
    - b. Find the slice that holds it: set $X_{t}\gets s_{k}$, where $(k-1)/N \leq F_{L}(g_{t}) < k/N$.
3. __Count:__ For each day $t = 2,3,\dots,T$ __do__:
    - a. Find the states of the two days: $X_{t-1} = s_{i}$ and $X_{t} = s_{j}$.
    - b. Count the move: $n_{ij}\gets n_{ij} + 1$.
4. __Transitions:__ Divide each row of counts by its sum: $\hat{p}_{ij}\gets n_{ij}/\sum_{k=1}^{N} n_{ik}$ for every pair of states $s_{i}$ and $s_{j}$.
5. __Emissions:__ For each state $s_{k}$ __do__:
    - a. Collect the growth rates of the days with $X_{t} = s_{k}$.
    - b. Set $\mu_{k}$ to their mean and $\sigma_{k}$ to their standard deviation.
6. __Start:__ Compute the stationary distribution $\bar{\pi}$ of $\hat{\mathbf{P}}$.

__Output__: The Laplace location $\mu$ and scale $b$, the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, and the stationary distribution $\bar{\pi}$.

__Notes:__
1. The Laplace cumulative distribution function has a closed form: $F_{L}(g) = \frac{1}{2}\exp\left((g-\mu)/b\right)$ for $g < \mu$, and $F_{L}(g) = 1 - \frac{1}{2}\exp\left(-(g-\mu)/b\right)$ for $g \geq \mu$.
2. Step 2 gives the same states as the slice edges $F_{L}^{-1}(k/N)$: a growth rate between two neighboring edges has a cumulative probability between $(k-1)/N$ and $k/N$.
3. `JumpHMM.jl` handles two rare cases. A state with no counted moves out of it gets the uniform row $\hat{p}_{ij} = 1/N$. A state with fewer than two days uses the mean and the standard deviation of all the growth rates.

The Student-t emissions and the wide extreme slices give the generator fat tails. To simulate a path, we draw a starting state from $\bar{\pi}$, step the chain with $\hat{\mathbf{P}}$, and draw each day's growth rate from the emission of the day's state.

### Why add jumps?
The counted chain forgets quickly: a few transitions usually return it from an extreme state to the middle states. The absolute growth rate of a simulated path stays correlated for only a few days, while real volatile periods last for weeks.

The model adds __jump episodes__ to keep volatile periods going. A jump episode starts with probability $\epsilon$ and lasts a Poisson number of days with mean $\lambda$. Each episode day draws its state from the most negative states with probability $p_{\text{neg}}$, slightly above one half, and from the most positive states otherwise:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

### Sampling algorithm

__Initialize__: Given the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, the degrees of freedom $\nu$, the stationary distribution $\bar{\pi}$, the episode probability $\epsilon$, the mean episode length $\lambda$, the number of tail states $N_{\text{tail}}$, the probability $p_{\text{neg}}$ that an episode day is negative, and the number of days $T$. 

The most negative states are $\mathcal{S}_{-} = \left\{s_{1},\dotsc,s_{N_{\text{tail}}}\right\}$, and the most positive states are $\mathcal{S}_{+} = \left\{s_{N-N_{\text{tail}}+1},\dotsc,s_{N}\right\}$. Draw the first state $X_{1}\sim\bar{\pi}$ and its growth rate $g_{1}$ from the emission of $X_{1}$. Set the number of episode days left $\texttt{remaining}\gets 0$.

For each day $t = 2,3,\dots,T$ __do__:
1. __New episode:__ If $\texttt{remaining} = 0$, __then__ draw $u\sim\text{Uniform}(0,1)$. If $u < \epsilon$, __then__ an episode starts: draw its length $\texttt{remaining}\sim\text{Poisson}(\lambda)$.
2. __State:__ Choose the state of day $t$:
    - If $\texttt{remaining} > 0$, __then__ day $t$ is an episode day. Draw $v\sim\text{Uniform}(0,1)$. If $v < p_{\text{neg}}$, __then__ draw $X_{t}$ uniformly from $\mathcal{S}_{-}$. Otherwise, draw $X_{t}$ uniformly from $\mathcal{S}_{+}$. Set $\texttt{remaining}\gets\texttt{remaining} - 1$.
    - __Else__, day $t$ is an ordinary day. Draw $X_{t}$ from the $\texttt{categorical}$ distribution defined by row $X_{t-1}$ of $\hat{\mathbf{P}}$.
3. __Emission:__ Draw the growth rate from the emission of the day's state: $g_{t}\gets\mu_{k} + \sigma_{k}\,T_{\nu}$, where $X_{t} = s_{k}$ and $T_{\nu}$ is a new Student-t draw.
4. __Store:__ Store $X_{t}$ and $g_{t}$.

__Output__: The states $\{X_{1},\dotsc,X_{T}\}$ and the growth rates $\{g_{1},\dotsc,g_{T}\}$ of one path.

__Notes:__
1. With $\epsilon = 0$ no episode starts, and the algorithm samples a hidden Markov model with no jumps.
2. A Poisson draw can be zero. In that case no episode starts, and day $t$ is an ordinary day.
3. The check in step 1 runs again on the day after an episode ends, so a new episode can follow at once.
4. `JumpHMM.jl` sets $N_{\text{tail}} = 5$ and $p_{\text{neg}} = 0.52$ unless we choose other values.

We tune the jump settings against the data in the SPY example. We then compare the simulated growth-rate distribution and volatility clustering with the data.

### What the episodes add and what they miss
Jump episodes add volatility clustering, but more or longer episodes thin the simulated tails and overstate the spread of growth rates. Each episode draws from the same tail states, keeping volatility near one level for about $\lambda$ days before switching off. In 2020, `SPY` volatility rose sharply before fading over several months.

The left panel sketches the two volatility profiles. The right shows the standard deviation of $g_{t}$ over the past 21 trading days for `SPY` from February 20, 2020, and the tuned model: six episodes and the mean over all episodes. Each period is aligned at its first day:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-Episode-Shape.svg" width="1000" style="max-width:100%; height:auto;" alt="Two panels with volatility on the vertical axis and trading days since the start on the horizontal axis. The left panel, a schematic: a dashed gray line marks the volatility of ordinary days. A red box, filled in light red, rises from the ordinary level to a middle level at the start, stays flat, and drops back after a red double arrow labeled about lambda days; it is labeled simulated episode, flat for about lambda days. A thick black curve labeled real crash (SPY, 2020), spikes, then fades, rises to a peak about twice as high as the box within the first few weeks, falls below the box level well before the box ends, and then fades slowly. The right panel, the data: the standard deviation of g over the past 21 trading days, in 1/yr, from 30 days before the start to 200 days after it. The black curve, real SPY from February 20, 2020, rises from about 2 to about 15 near day 30 and falls back to about 4 by day 60. The thick red curve, the mean over the episodes of the tuned model, rises to about 7 within a month, stays near 7 until about day 100, and falls to about 2.6 by day 150. Six thin red curves, single episodes, have the same flat shape at levels between about 5 and 10. A dashed line marks day 0."/>
    </center>
</div>

A stress test on these paths meets long, moderate volatile periods rather than sharp crashes that fade.

> __Example:__
>
> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024. We compare simulated distributions and volatility clustering with the data, then tune jump episodes and examine what they change.

Next, let's extend the generator from one asset to many.

___

## Many Assets: The Sector Factor Model
Previously, we studied __factor models__, which explain __co-movement__: how stocks move together. For stock $i$ on day $t$, a general factor model is given by:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{j}\delta_{ij}\,f_{j}(t)}_{\text{other factors}} + \underbrace{\varepsilon_{i}(t)}_{\text{residual}}
$$

Here $\alpha_{i}$ is the stock-specific baseline (intercept), $g_{m}(t)$ is market growth, and $f_{j}(t)$ are other factors, such as interest rates or inflation. The coefficients $\beta_{i}$ and $\delta_{ij}$ measure sensitivity to these factors. The __residual__ $\varepsilon_{i}(t)$ is the growth left after subtracting the baseline and factor contributions.

> __Definition (sector factor model):__ Each stock has one other factor besides the market, the factor $f_{s}(t)$ of its sector $s$. This factor captures co-movement left unexplained by the market.
>
> The daily growth rate of stock $i$ in sector $s$ is given by:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{\psi_{i}\,\tilde{\varepsilon}_{i}(t)}_{\text{stock residual}}
> $$
>
> The four components are:
> * __Baseline:__ The stock-specific baseline $\alpha_{i}$ is the intercept from the stock's single index fit. We hold $\alpha_{i}$ and the market sensitivity $\beta_{i}$ fixed.
> * __Market:__ The market draw $g_{m}(t)$ comes from a jump hidden Markov model of `SPY`.
> * __Sector:__ The sector draw $f_{s}(t)$ comes from a jump hidden Markov model of the average single index residual of the stocks in sector $s$. The loading $\delta_{i}$ is the slope of the stock's residual on the other residuals in its sector, scaled to match the sector's co-movement in the data.
> * __Stock-specific residual:__ The component $\psi_{i}\,\tilde{\varepsilon}_{i}(t)$ models moves left after removing the baseline, market, and sector contributions, such as an earnings surprise. We center a draw from a jump hidden Markov model of this remaining residual and size it with $\psi_{i}$.
>
> On each day, every stock shares the market draw, and every stock in a sector shares the sector draw. Each stock's residual path is sampled independently of the market, the sectors, and the other stocks' residual paths. The shared market and sector draws are the only link between stocks.

Let's write out how we construct the stock-specific residual component:

> __The stock-specific residual component (fixed β):__
>
> In the data, we subtract the market fit from the observed growth rate. We then subtract the sector contribution:
>
> $$
> \begin{aligned}
> \varepsilon_{i}(t) &= g_{i}(t)-\alpha_{i}-\beta_{i}g_{m}(t)
> && \text{residual after the market fit} \\
> u_{i}(t) &= \varepsilon_{i}(t)-\delta_{i}f_{s}(t)
> && \text{residual after market and sector}.
> \end{aligned}
> $$
>
> We fit a jump hidden Markov model to the series $u_{i}(t)$. Let $u_{i}^{\mathrm{draw}}(t)$ be a draw from that model and $\bar{u}_{i}$ its long-run mean. We center the draw on zero:
>
> $$
> \tilde{\varepsilon}_{i}(t) = u_{i}^{\mathrm{draw}}(t)-\bar{u}_{i}.
> $$
>
> The simulated sector and stock residual paths are independent. Their variances add. We use the variances in the real data to set a target for their sum:
>
> $$
> \underbrace{\delta_{i}^{2}\operatorname{Var}(f_{s})}_{\text{sector variance}}
> + \underbrace{\psi_{i}^{2}\operatorname{Var}(u_{i})}_{\text{target stock residual variance}}
> = \underbrace{\operatorname{Var}(\varepsilon_{i})}_{\text{variance left after the market fit}}.
> $$
>
> Solving for the scale gives:
>
> $$
> \boxed{\psi_{i}^{2} =
> \frac{\operatorname{Var}(\varepsilon_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}
> {\operatorname{Var}(u_{i})}}
> \qquad
> \psi_{i} \geq 0.
> $$
>
> The sector factor includes stock $i$ in its average. The remaining series $u_{i}$ can still move with that factor in the data. The scale compensates for the covariance lost when we draw the two paths independently.
>
> The variance equation sets a target. The fitted jump hidden Markov models reproduce the data variances only approximately.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the jump hidden Markov models of the market, of each sector's factor, and of each stock's residual after its sector term; each stock's single index estimates $\alpha_{i}$ and $\beta_{i}$ and sector loading $\delta_{i}$; the scale of each stock's residual component, $\psi_{i}^{2} = \left(\operatorname{Var}(\varepsilon_{i}) - \delta_{i}^{2}\operatorname{Var}(f_{s})\right)/\operatorname{Var}(\varepsilon_{i}-\delta_{i}f_{s})$, with the variances of the real data; each stock's real starting price $S_{i}(0)$; and the number of days $T$. Draw a market path $g_{m}(1),\dotsc,g_{m}(T)$ from the market model. For each sector $s$, draw a factor path $f_{s}(1),\dotsc,f_{s}(T)$ from the model of sector $s$, independently of the other sectors and of the market.

For each stock $i$ in sector $s$ __do__:
1. __Residual draw:__ Draw a path from the model of the stock's remaining residual and subtract the model's long-run mean (the mean of one long simulated path) to get $\tilde{\varepsilon}_{i}(1),\dotsc,\tilde{\varepsilon}_{i}(T)$.
2. __Compose:__ For each day $t = 1,2,\dots,T$ __do__:
    - a. Compose the growth rate: $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{\varepsilon}_{i}(t)$.
    - b. Compound the price, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Notes:__
1. The sector factor averages the residuals of the stocks in sector $s$, stock $i$ included. Independent draws drop the part of the stock's leftover residual that moves with the sector factor, and $\psi_{i}$ puts that part back. In a sector with many stocks, $\psi_{i}$ is close to one.
2. With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 2. We also reduce $\psi_{i}$ to preserve the variance target.

### An option: a drifting β
The coefficient  measures stock $i$'s sensitivity to market growth. Holding $\beta_{i}$ fixed assumes that stock $i$'s sensitivity to market growth rate stays constant throughout a synthetic path. Alternatively, we can let the sensitivity change over time around its single index estimate $\hat{\beta}_{i}$. 

Each day, we update it with:

$$
\beta_{i}(t) = \hat{\beta}_{i}
+ \underbrace{(1-a)\left(\beta_{i}(t-1)-\hat{\beta}_{i}\right)}_{\text{retained deviation}}
+ \underbrace{a\,k_{i}\,z_{i}(t)}_{\text{new shock}}.
$$

Here $0<a<1$ is the exponential moving average weight. A small $a$ retains most of the previous day's deviation from $\hat{\beta}_{i}$. The shocks $z_{i}(t)$ are independent standard normal draws, with their size controlled by $k_{i}\geq 0$. In the example, we choose $k_{i}$ using the variation in β estimates updated daily from real data. These estimates also contain sampling noise.

The market contribution is now $\beta_{i}(t)g_{m}(t)$. This lets us test whether a trading method can track changes in a stock's market sensitivity.

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.

___

## Optional Advanced Material
> [▶ Review Markov chains and hidden Markov models](advanced/markov/CHEME-5660-L13a-Lecture-MarkovModels-Fall-2026.ipynb). In this notebook, we define the Markov property and the transition matrix, compute the stationary distribution by power iteration, and sample a hidden Markov model. Its two examples apply a hidden Markov model to a person's mood and to the daily growth rate of `SPY`.

___

## Summary
This lecture builds synthetic markets for stress tests, from Markov chains and hidden Markov models to a jump hidden Markov model of one asset and a sector factor model of many stocks.

> __Key Takeaways:__
>
> * __Markov chains and hidden states:__ We defined the state space, the transition matrix, and the stationary distribution, and we estimated the transition matrix by counting transitions. Hidden states extend the chain, and their emission distributions produce the values we observe.
> * __Jump hidden Markov models reproduce fat tails and some volatility clustering:__ We binned growth rates into states of equal probability, counted transitions, and used Student-t emissions. We added and tuned jump episodes to prolong volatile periods. Episodes add clustering, but thin the tails and stay flat where a real crash spikes and fades.
> * __Shared draws make synthetic stocks move together:__ We combined each stock's baseline with shared market and sector components and a stock-specific residual component. We sized the residual component to preserve the variance left after the market fit when combined with the sector component.

Synthetic markets let us test a trading method on many plausible paths, including sell-offs that never happened, rather than on one historical path.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___